In [ ]:
import pandas as pd
import pymysql
import unicodedata
import re
from nltk.corpus import stopwords
import string
from collections import Counter

In [5]:
conexion = pymysql.connect(
    host="dl-radar.cluster-ro-c7pmwdslewrp.us-east-1.rds.amazonaws.com",
    user = "debian",
    password= "eeAZU3v1FXCY9zmbvcS6kpEpyj",
    database="data_fact",
    port= 4408
)

cursor = conexion.cursor()

In [6]:
query = """
SELECT *
FROM base_rucs_sri;
"""
base_registro_civil = pd.read_sql_query(query, conexion)

C:\Users\anali\AppData\Local\Temp\ipykernel_29500\631102470.py:5: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  base_registro_civil = pd.read_sql_query(query, conexion)


In [4]:
base_registro_civil = pd.read_parquet(r"C:\Users\anali\OneDrive - PUBLIPROMUEVE S.A\Ruben Freire's files - CENTROS COMERCIALES\sandbox\bases\base_rucs_sri.parquet")

In [44]:
def quitar_tildes(texto):
    return ''.join(
        c for c in unicodedata.normalize('NFD', texto)
        if unicodedata.category(c) != 'Mn'
    )

def limpiador(lista: list):
    documentos = []
    re_punctuation = re.compile('[%s]' % re.escape(string.punctuation))

    stop_words_s = set(stopwords.words('spanish'))
    stop_words = set(stopwords.words('english'))

    for descripcion in lista:
        tokens = descripcion.split()
        tokens = [re_punctuation.sub(' ', w) for w in tokens]
        tokens = ' '.join(tokens).split()

        tokens = [quitar_tildes(word.lower()) for word in tokens]

        tokens = [word.lower() for word in tokens if re.search('[a-z ]', word.lower())]
        tokens = [word.lower() for word in tokens if word.isalpha()]
        
        tokens = [w for w in tokens if w not in stop_words_s]
        tokens = [w for w in tokens if w not in stop_words]
        
        tokens = [word for word in tokens if len(word) > 2]

        if len(tokens) > 0:
            documento = ' '.join(tokens)
        else:
            documento = ''  # ← asegura longitud igual al DF

        documentos.append(documento)

    return documentos

In [45]:
base_registro_civil['nombre_fantasia_comercial'] = (
    base_registro_civil['nombre_fantasia_comercial']
        .replace("", pd.NA)
)


In [46]:
base_registro_civil = base_registro_civil[base_registro_civil["nombre_fantasia_comercial"].notna()] 

In [47]:
base_registro_civil['motivo_cancelacion_suspension'] = (base_registro_civil['motivo_cancelacion_suspension'].replace("", pd.NA))

C:\Users\anali\AppData\Local\Temp\ipykernel_20400\3861836433.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  base_registro_civil['motivo_cancelacion_suspension'] = (base_registro_civil['motivo_cancelacion_suspension'].replace("", pd.NA))


In [48]:
base_registro_civil = base_registro_civil[base_registro_civil['motivo_cancelacion_suspension'].isna()]

In [144]:
recreo_nombres = pd.read_excel(r"C:\Users\anali\OneDrive - PUBLIPROMUEVE S.A\Ruben Freire's files - CENTROS COMERCIALES\ccrecreo.xlsx")

In [145]:
len(recreo_nombres)

541

In [146]:
dicc_nombre_fantasia = set(recreo_nombres['LOCAL'])

In [147]:
#Normalizamos los q gramas
def normalizar(s):
    if pd.isna(s):
        return ""
    s = s.lower()
    s = unicodedata.normalize('NFD', s)
    s = ''.join(c for c in s if unicodedata.category(c) != 'Mn')
    s = re.sub(r'[^a-z0-9 ]', ' ', s)
    s = re.sub(r'\s+', ' ', s).strip()
    return s

In [148]:
#parte una palabra en la cantidad de q gramas dados
def qgrams(s, q=3):
    return {s[i:i+q] for i in range(len(s) - q + 1)}

In [149]:
#Es la medida que vamos a tomar. La cantidad de q gramas dados los que coinciden dividido para todos
def jaccard(a, b):
    if not a or not b:
        return 0.0
    return len(a & b) / len(a | b)


In [150]:
# Normalizamos los nombres fantasia
dic_norm = {normalizar(x): x for x in dicc_nombre_fantasia}

dic_qgrams = {
    k: qgrams(k, q=3)
    for k in dic_norm.keys()
}

In [151]:
#Match de los qgramas en los centros comerciales
def match_qgram(nombre,centro_comercial, threshold=0.5, umbral_centro_comercial = 0.28):
    s = normalizar(nombre)
    q_s = qgrams(s)

    mejor, score = None, 0
    for k, q_k in dic_qgrams.items():
        sim = jaccard(q_s, q_k)
        if sim > score:
            mejor, score = dic_norm[k], sim

    if score >= threshold:
        return mejor, score
    elif (score>= umbral_centro_comercial) and (centro_comercial in nombre.lower()):
        return mejor, score 
    return None, score

In [152]:
#Normalizamos el nombre fantasía comercial
base_registro_civil['nombre_fantasia_comercial'] = base_registro_civil['nombre_fantasia_comercial'].apply(normalizar)

#Tenemos la direccion completa separada en base registro civil
base_registro_civil[['provincia',  'canton', 'parroquia', 'calles']] = (
    base_registro_civil['direccion_completa']
        .str.split('/', n=3, expand=True)
)

In [102]:
#Para sacarnos las frecuencias

#Nombre del centro comercial
nombre = 'san'
apellido = 'marino'
rejex = f'(?=.*{nombre})(?=.*{apellido})'

#Filtros para las calles
mask_regex = base_registro_civil['nombre_fantasia_comercial'].str.contains(rf'{rejex}', case = False, na = False)
#mask_canton = base_registro_civil['canton'].str.contains("Quito", case = False, na =  False)
#mask_parroquia =  base_registro_civil['parroquia'].str.contains("magda", case = False, na =  False)

#Filtramos la base para obtener las calles
base_filtrada = base_registro_civil[mask_regex]

#Obtenemos las calles
list_calles =list(base_filtrada['calles'])

#Limpiamos las calles y hacemos una sola list
list_calles_limpia = limpiador(list_calles)
list_calles_limpia_total = [
    palabra
    for i in range(len(list_calles_limpia))
    for palabra in list_calles_limpia[i].split()
]

#Sacamos las frecuencias
frecuencias = Counter(list_calles_limpia_total)

#Hacemos dataframe de frecuencias
df_frecuencias = ( 
    pd.DataFrame(frecuencias.items(), columns = ['palabra', 'frecuencia'])
    .sort_values('frecuencia', ascending = False)
    .reset_index(drop = True)
)


In [103]:

#Guardamos las frecuencias
df_frecuencias.to_excel(rf"frecuencias_{nombre}_{apellido}.xlsx", index = False)

In [153]:
mask_canton = base_registro_civil['canton'].str.contains("QUITO", case = False, na = False) 
mask_parroquia = base_registro_civil['parroquia'].str.contains("magdalena", case = False, na = False)
#mask_calle_mald  = base_registro_civil['calles'].str.contains("", case = False, na = False)
mask_calles = base_registro_civil['calles'].str.contains("pedro|maldonado|vicente|lauro|guerrero|becerra|s11c|oe2f", case = False, na = False)

mask_general = (mask_canton & mask_parroquia) | mask_calles

In [154]:
base_registro_civil_test = base_registro_civil[mask_general]

In [159]:
base_registro_civil_test[['match_qgram', 'score_qgram']] = (
    base_registro_civil_test['nombre_fantasia_comercial']
      .apply(lambda x: pd.Series(match_qgram(x, centro_comercial='recreo', threshold=0.55, umbral_centro_comercial=0.25)))
)

In [160]:
base_registro_civil_test = base_registro_civil_test.sort_values(by = 'score_qgram', ascending = False)

In [161]:
base_registro_civil_test[base_registro_civil_test['match_qgram'].notna()][['numero_ruc','match_qgram', 'score_qgram', 'nombre_fantasia_comercial']]

,numero_ruc,match_qgram,score_qgram,nombre_fantasia_comercial
7184735,1.791360e+12,TEXAS CHICKEN,1.00,texas chicken
7189821,1.791410e+12,MIL COLORES,1.00,mil colores
824376,5.025639e+11,TINTALANDIA,1.00,tintalandia
3641027,1.001223e+12,MODA SPORT,1.00,moda sport
3588063,9.933859e+11,CRECOS,1.00,crecos
...,...,...,...,...
7126437,1.790398e+12,MEGAMAXI,0.25,prontto megamaxi el recreo
7143912,1.790710e+12,FYBECA,0.25,fybeca metrorecreo
7158303,1.790995e+12,CRECOS,0.25,recreo 2
7391311,1.793229e+12,CRECOS,0.25,pecos shoes recreo
